<a href="https://colab.research.google.com/github/ArkanDash/Advanced-RVC-Inference/blob/master/notebook/train-noui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


<h1><font color="#0000">📳 Advanced RVC Inference Training Only</font></h1>


<hr>
<p>
  <a href="https://discord.gg/hvmsukmBHE"><font color="#a78bfa">Discord</font></a> ·
  <a href="https://github.com/ArkanDash/Advanced-RVC-Inference"><font color="#a78bfa">Github</font></a> ·
  <a href="https://github.com/ArkanDash/Advanced-RVC-Inference/blob/master/docs/README.md"><font color="#a78bfa">CLI Guide</font></a> ·
  <a href="https://colab.research.google.com/github/ArkanDash/Advanced-RVC-Inference/blob/master/Advanced-RVC.ipynb"><font color="#a78bfa">Main Colab</font></a>
</p>

<h2><font color="#9ca3af">⚙️ Setup</font></h2>
<p>Mount Google Drive and clone the repo locally. This notebook skips the Gradio web UI for a lighter, faster setup.</p>


In [ ]:
# @title Mount Google Drive
from google.colab import drive
from google.colab._message import MessageError

try:
  drive.mount("/content/drive")
except MessageError:
  print("❌ Failed to mount drive")

In [ ]:
# @title Install Deps


from IPython.display import clear_output
import os, sys, subprocess

REPO_DIR = "/content/Advanced-RVC-Inference"
LOGS_PATH = f"{REPO_DIR}/arvc/assets/logs"
BACKUPS_PATH = "/content/drive/MyDrive/RVCBackup"
CLI = "python -m arvc.api.cli"

def run(cmd, label, quiet=True):
    """Run a shell command, show a clean status line, suppress noise."""
    print(f"[1/4] {label}...", end=" ", flush=True)
    try:
        subprocess.run(cmd, shell=True, check=True,
                       capture_output=quiet, text=True)
        print("OK")
    except subprocess.CalledProcessError as e:
        print(f"FAILED ({e.returncode})")
        if quiet and e.stderr:
            print(f"       {e.stderr.strip().splitlines()[-1][:200]}")

# 1. System deps (apt)
run("apt-get -y install libportaudio2 ffmpeg git -qq", "system packages")

# 2. uv (faster pip)
run("pip install uv -q", "uv package manager")

# 3. Clone or pull the repo
if os.path.exists(REPO_DIR):
    %cd $REPO_DIR
    run("git pull origin master -q", "git pull (latest security patches)")
else:
    run(f"git clone https://github.com/ArkanDash/Advanced-RVC-Inference.git {REPO_DIR} -q",
        "git clone")
    %cd $REPO_DIR

# 4. Python deps
run("uv pip install -r requirements.txt --system -q 2>/dev/null || uv pip install -r requirements.txt --system -q",
    "Python deps")
# ── NUMPY COMPATIBILITY PATCH (Colab ABI fix) ─────────# Colab sometimes ships numpy 2.x which breaks torch/scipy/numba# (ABI mismatch: dtype size 88 vs 96). Force numpy <2.0.0.
_numpy_restart_needed = False
print("[4.5] Numpy compatibility check...", end=" ", flush=True)
try:
    import numpy as _np
    _v = _np.__version__
    # Check if numpy 2.x (incompatible with this project)
    if _v.startswith('2.'):
        print(f"downgrading {_v} -> 1.x...")
        subprocess.run("pip install --force-reinstall 'numpy>=1.25.2,<2.0.0' -q",
            shell=True, check=True, capture_output=True
        )
        # Clear any cached compiled extensions with wrong ABI
        subprocess.run("find /usr/local/lib/python*/site-packages -name '_*.so' -delete 2>/dev/null; true",
            shell=True, check=False, capture_output=True
        )
        print("OK (downgraded)")
        _numpy_restart_needed = True
    else:
        print(f"OK ({_v} compatible)")
except Exception as _e:
    print(f"WARN ({_e})")
    # Try force-install anyway
    subprocess.run("pip install --force-reinstall 'numpy>=1.25.2,<2.0.0' -q 2>/dev/null || true",
        shell=True, check=False, capture_output=True
    )
    _numpy_restart_needed = True
# ── AUTO RESTART (if numpy was downgraded) ─────────────# After downgrading numpy, we MUST restart the runtime because:#   1. Python still has old numpy 2.x .so files loaded in memory#   2. torch/scipy/numba were compiled against different ABI#   3. Without restart: ValueError: numpy.dtype size changed
if _numpy_restart_needed:
    print("\n" + "=" * 60)
    print("  [!] NUMPY DOWNGRADED - RESTARTING RUNTIME AUTOMATICALLY...")
    print("      The installation will continue after restart.")
    print("      Please wait ~10 seconds for reconnect...")
    print("=" * 60)
    import time
    time.sleep(2)
    # Method 1: Kill process (triggers Colab auto-restart)
    os.kill(os.getpid(), 9)
# 5. ONNX Runtime — pin a CUDA-12-compatible version to avoid the
# libcudart.so.13 ImportError that breaks the UVR5 separator chain.
# Colab ships CUDA 12; onnxruntime-gpu>=1.21 wants CUDA 13.
print("[5/4] ONNX runtime (pinned for Colab CUDA 12)...", end=" ", flush=True)
try:
    subprocess.run(
        "uv pip install --system -q 'onnxruntime-gpu==1.20.1' 2>/dev/null "
        "|| pip install -q 'onnxruntime-gpu==1.20.1' 2>/dev/null "
        "|| pip install -q onnxruntime",
        shell=True, check=False, capture_output=True, text=True
    )
    print("OK")
except Exception as e:
    print(f"WARN ({e})")

# Ensure the package itself is importable
%cd $REPO_DIR
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ── Verify install ─────────────────────────
_verify_results = []

# Security patches
try:
    from arvc.engine.models.safe_load import (
        safe_torch_load, safe_pickle_load, safe_yaml_load,
        validate_path_within, safe_onnxruntime_import
    )
    _verify_results.append(("Security patches (safe_load)", True, ""))
except ImportError as e:
    _verify_results.append(("Security patches (safe_load)", False, str(e)))

# ONNX runtime
try:
    import onnxruntime as _ort
    _providers = _ort.get_available_providers()
    _verify_results.append((
        f"ONNX runtime ({_ort.__version__})",
        True,
        f"providers: {', '.join(_providers[:3])}"
    ))
except ImportError as e:
    _verify_results.append((
        "ONNX runtime",
        False,
        str(e)[:120]
    ))

# PyTorch + CUDA
try:
    import torch
    if torch.cuda.is_available():
        _gpu = torch.cuda.get_device_name(0)
        _verify_results.append((
            f"PyTorch {torch.__version__} + CUDA",
            True,
            f"GPU: {_gpu}"
        ))
    else:
        _verify_results.append((
            f"PyTorch {torch.__version__}",
            True,
            "CPU mode (no GPU detected)"
        ))
except ImportError as e:
    _verify_results.append(("PyTorch", False, str(e)))

clear_output()

# ── Clean summary output ───────────────────────
print()
print("Advanced RVC Inference — Install Summary")
print("=" * 50)
for name, ok, detail in _verify_results:
    icon = "OK  " if ok else "FAIL"
    print(f"  [{icon}] {name}")
    if detail:
        print(f"         {detail}")
print("=" * 50)
print()
print("Available CLI flags:")
print("  --fast_train    ~3x speedup (TF32+cuDNN+torch.compile, vocal-quality-safe)")
print("  --bf16_adamw    Applio-parity bf16 shortcut (A100/H100 only, skip on T4)")
print()
print("Next steps:")
print("  python -m arvc.api.cli --help")
print("  python -m arvc.api.cli info")
print()
all_ok = all(ok for _, ok, _ in _verify_results)
if all_ok:
    print("Status: READY")
else:
    print("Status: PARTIAL — some components failed, see above")

In [ ]:
# @title Set Environment Variables
# @markdown Tune these for better performance or compatibility.
import os
import sys

# Ensure working directory is always the repo root
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
# SECURITY + PERF PATCH: was '1' — forced synchronous CUDA ops and killed
# training throughput. Set to '0' so --fast_train can actually deliver its
# ~3x speedup. Set to '1' only for debugging CUDA errors.
os.environ['CUDA_LAUNCH_BLOCKING'] = '0'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['FAISS_DISABLE_CPU_FEATURES'] = 'AVX512,AVX2,AVX512_SPR,SVE'

print("✅ Environment variables set.")
print(f"  TF_CPP_MIN_LOG_LEVEL = {os.environ['TF_CPP_MIN_LOG_LEVEL']}")
print(f"  PYTORCH_ENABLE_MPS_FALLBACK = {os.environ['PYTORCH_ENABLE_MPS_FALLBACK']}")
print(f"  CUDA_LAUNCH_BLOCKING = {os.environ['CUDA_LAUNCH_BLOCKING']}  (0 = async CUDA, required for --fast_train)")
print(f"  Working directory: {os.getcwd()}")

<h2><font color="#38ef7d">📋 System Info</font></h2>
<p>Check your GPU, installed models, and available F0 methods.</p>


In [ ]:
# @title System Info
import os, sys
os.chdir(REPO_DIR)
!python -m arvc.api.cli info

In [ ]:
# @title List Installed Models
import os
os.chdir(REPO_DIR)
!python -m arvc.api.cli list-models

In [ ]:
# @title List F0 Methods
import os
os.chdir(REPO_DIR)
!python -m arvc.api.cli list-f0-methods

In [ ]:
# @title Show Version
import os
os.chdir(REPO_DIR)
!python -m arvc.api.cli version

In [ ]:
# @title Download Pretrained Models
# @markdown Download pretrained RVC base models needed for training.
# @markdown Available models are fetched from the community pretrained JSON.
pretrained_model = "KLM4.3_X3"  # @param ["DMRV1", "DMRV2", "GuideVocalPretrain", "IMA", "Itaila", "KLM4.0", "KLM4.1", "KLM4.2", "KLM4.3_X1", "KLM4.3_X2", "KLM4.3_X3", "KLM4.3_X4", "KLM4.9_HFG", "KLM_BeatMaster", "KLM_BeatzForge", "NanashiV1", "NanashiV1.5", "NanashiV1.7", "NanashiV2Base", "NanashiV2Finetune", "Nanashi_Anime_Normal", "Nanashi_Anime_Resize", "OV2Super", "RIN_E3", "RigelV1.5", "Rigel_Base", "Rigel_FineTuned", "SingerPretrain", "Snowie-X-RinE3", "SnowieRuPretrain", "SnowieV3.1", "Titan_Medium", "UKA"]
pretrained_sr = "48k"  # @param ["32k", "40k", "48k"]

import os, json, requests
os.chdir(REPO_DIR)

# Fetch available pretrained data from the community JSON
pretrained_url = "https://huggingface.co/buckets/R-Kentaren/Ultimate-RVC-Models/resolve/json/custom_pretrained.json"
try:
    resp = requests.get(pretrained_url, timeout=15)
    resp.raise_for_status()
    pretrained_data = resp.json()
except Exception as e:
    print(f"❌ Failed to fetch pretrained list: {e}")
    pretrained_data = {}

# Validate selection
if pretrained_model not in pretrained_data:
    print(f"❌ Model '{pretrained_model}' not found in pretrained list.")
    print(f"Available: {', '.join(sorted(pretrained_data.keys()))}")
elif pretrained_sr not in pretrained_data[pretrained_model]:
    available_sr = list(pretrained_data[pretrained_model].keys())
    print(f"❌ Sample rate '{pretrained_sr}' not available for '{pretrained_model}'.")
    print(f"Available rates: {', '.join(available_sr)}")
else:
    from arvc.services.downloads import download_pretrained_model
    from arvc.utils.variables import translations

    result = download_pretrained_model(
        choices=translations["list_model"],
        model=pretrained_model,
        sample_rate=pretrained_sr
    )
    print(result if result else "✅ Pretrained models downloaded.")

<h2><font color="#fee140">🏋️ Training Pipeline</font></h2>

<details><summary><b>🔒 Security + 🚀 Speedup + 🎯 Accuracy notes</b></summary>
<br>

**Security patches (active on this Colab)** — all commits up to `f06a392`:
- All `torch.load(...)` calls now route through `safe_torch_load` (forces `weights_only=True`). 16 model-loading paths hardened across predictors (PESTO, PENN, RMVPE, CREPE, DJCM, FCPE×2), training (train.py × 3, data_utils, utils), whisper, onnx_export, vr_separator, fairseq.
- Restricted `pickle.Unpickler` whitelist (only primitive + numpy types) — blocks every known pickle RCE gadget (`os.system`, `subprocess.Popen`, `builtins.eval`).
- `yaml.safe_load` everywhere — never `FullLoader`.
- `validate_path_within()` wired into 20+ `os.path.join` sites in `inference.py` and `services/training.py` — blocks `../../etc/cron.d/evil` style path-traversal from GUI/CLI inputs.
- All 5 downloaders (HuggingFace, Google Drive, Mega, MediaFire, PixelDrain) now: hard 8 GB size cap, extension whitelist (`.pth/.pt/.onnx/.index/.zip/...`), filename sanitization (forces single basename component), `timeout=300s` on every network call.
- `tempfile.mktemp` → `tempfile.mkstemp` in `gdown.py` — closes the classic TOCTOU symlink race.
- `random.randint` → `secrets.randbits(32)` for the MEGA nonce.
- Bare `except:` in `train.py:897` (silent checkpoint-load failure → silent restart from epoch 1) → typed `except (FileNotFoundError, RuntimeError, OSError, KeyError, ValueError)` with a logged warning. Same fix in `onnx_export.py:115`.
- `urllib.request.urlopen` (Google Sheets fetch at app startup) now has `timeout=30` so a hung response can't hang the whole app.

**🚀 Speedup — `--fast_train` flag (vocal-quality-safe, ~3x faster)**:
- TF32 matmul + cuDNN TF32 (Ampere+ GPUs — RTX 30xx/40xx/A100/H100). 10-bit mantissa is well below the audible noise floor for vocal training.
- cuDNN `benchmark=True` + `deterministic=False` — picks the fastest conv kernel per input shape.
- `torch.compile(mode="reduce-overhead")` on both G and D — fuses kernels, uses CUDA graphs.
- `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,max_split_size_mb:512` — avoids fragmentation on long runs.
- DataLoader: `num_workers=8`, `prefetch_factor=16`, `pin_memory=True`, `persistent_workers=True`.
- All of the above are **non-numerical** optimizations — no loss function, gradient path, or weight is touched. Bit-for-bit identical vocal fidelity.

**🚀 Speedup — `--bf16_adamw` flag (Applio-parity shortcut)**:
- Forces `optimizer=AnyPrecisionAdamW` and `brain=True` (bf16 autocast).
- bf16 has the same exponent range as fp32 (no overflow like fp16), so it's safe to use without a GradScaler.
- On Ampere/Hopper, bf16 matmul is ~2x faster than fp32. Combined with `--fast_train`, this is the single biggest "free" speedup.
- Recommended on Colab A100 / H100. On T4 (Turing), bf16 is emulated — use plain `--fast_train` instead.

**🎯 Accuracy — Applio parity for 10-minute datasets**:
- `per_preprocess` was 3.7s, now 3.0s — matches Applio's `PERCENTAGE=3.0`. This produces **~26% more training chunks** for the same audio, which is the largest single cause of "ARVC less accurate than Applio on small data".
- `--chunk_len` / `--overlap_len` CLI flags now apply to **Automatic** cut mode (was Simple-only). For a 10-min dataset, try `--overlap_len=0.5` to extract ~17% more chunks.
- `preprocess.py` now writes `total_dataset_duration` + `total_seconds` to `model_info.json` (Applio parity).
- `extract_model()` now embeds `embedder_model`, `dataset_length`, and `overtrain_info` into the saved `.pth` as provenance metadata — lets inference auto-select the matching embedder.
- Preprocess now fails fast with a clear error if the dataset path is missing or empty (was a silent walk + cryptic downstream crash).

**What's NOT changed** (so you don't lose anything):
- Multi-scale mel loss still defaults ON (8 scales, dynamic windows — better than Applio's 7 scales).
- 15+ F0 methods still available (Applio has 3).
- 5+ optimizers (Applio has 2), gradient accumulation, cosine LR, all backends (CUDA/DirectML/OpenCL/ZLUDA/XPU) still supported.

</details>



In [ ]:
!mkdir -p dataset

In [ ]:
# @markdown ---
# @markdown ### 📦 Dataset Settings
# @title Step 1: Create Dataset
# @markdown Build a training dataset from YouTube URLs or local audio files.
source = ""  # @param {type:"string"}
output_dir = "dataset"  # @param {type:"string"}
sample_rate = 40000  # @param ["32000", "40000", "48000"]
clean_dataset = False  # @param {type:"boolean"}
clean_strength = 0.7  # @param {type:"slider", min:0, max:1, step:0.01}
separate_vocals = False  # @param {type:"boolean"}
separator_model = "MDXNET_Main"  # @param {type:"string"}
separator_reverb = False  # @param {type:"boolean"}
reverb_model = "MDX-Reverb"  # @param {type:"string"}
skip_start = 0  # @param {type:"slider", min:0, max:300, step:1}
skip_end = 0  # @param {type:"slider", min:0, max:300, step:1}

import os
os.chdir(REPO_DIR)

# Auto-detect if source is a URL or local path
if source.startswith("http"):
    cmd = f"python -m arvc.api.cli create-dataset -u \"{source}\" -o \"{output_dir}\" --sample_rate {sample_rate}"
else:
    cmd = f"python -m arvc.api.cli create-dataset -i \"{source}\" -o \"{output_dir}\" --sample_rate {sample_rate}"

if clean_dataset:
    cmd += f" --clean_dataset --clean_strength {clean_strength}"
if not separate_vocals:
    cmd += " --no-separate"
else:
    if separator_reverb:
        cmd += " --separate_reverb"
    cmd += f" --separator_model {separator_model}"
    if separator_reverb:
        cmd += f" --reverb_model {reverb_model}"
if skip_start > 0:
    cmd += f" --skip_start {skip_start}"
if skip_end > 0:
    cmd += f" --skip_end {skip_end}"

!{cmd}

In [ ]:
# @markdown ---
# @markdown ### ✂️ Slicing Settings
# @title Step 2: Preprocess
# @markdown Slice and normalize training audio data.
# @markdown **Recommended**: Enable process_effects (high-pass filter) and use 'post' normalization for best results.
# @markdown **Applio-parity (commit f06a392+)**: `--chunk_len`/`--overlap_len` now apply to **Automatic** cut mode too,
# @markdown not just Simple. Default chunk_len=3.0 matches Applio's PERCENTAGE for ~26% more chunks on 10-min datasets.
model_name = ""  # @param {type:"string"}
sample_rate = 40000  # @param ["32000", "40000", "48000"]
dataset_path = "./dataset"  # @param {type:"string"}
cpu_cores = 2  # @param {type:"slider", min:1, max:8, step:1}
cut_method = "Automatic"  # @param ["Automatic", "Simple", "Skip"]
process_effects = True  # @param {type:"boolean"}
clean_dataset = False  # @param {type:"boolean"}
clean_strength = 0.7  # @param {type:"slider", min:0, max:1, step:0.01}
chunk_len = 3.0  # @param {type:"slider", min:0.5, max:10, step:0.5}
overlap_len = 0.3  # @param {type:"slider", min:0, max:1, step:0.05}
normalization = "post"  # @param ["none", "pre", "post"]

import os
os.chdir(REPO_DIR)

cmd = f"python -m arvc.api.cli preprocess {model_name} --sample_rate {sample_rate}"
cmd += f" --dataset_path \"{dataset_path}\" --cpu_cores {cpu_cores}"
cmd += f" --cut_method {cut_method}"
if process_effects:
    cmd += " --process_effects"
if clean_dataset:
    cmd += f" --clean_dataset --clean_strength {clean_strength}"
# SECURITY/ACCURACY PATCH (commit f06a392+): always pass --chunk_len / --overlap_len.
# These now apply to Automatic cut mode too (previously only Simple). For 10-min
# datasets, try overlap_len=0.5 to extract ~17% more training chunks.
cmd += f" --chunk_len {chunk_len} --overlap_len {overlap_len}"
if normalization != "none":
    cmd += f" --normalization {normalization}"

!{cmd}

In [ ]:
# @markdown ---
# @markdown ### 🔬 Feature Extraction Settings
# @title Step 3: Extract Features
# @markdown Extract embeddings and F0 features from preprocessed data.
# @markdown **Recommended**: Use rmvpe for best F0 extraction quality.
model_name = ""  # @param {type:"string"}
sample_rate = 40000  # @param ["32000", "40000", "48000"]
f0_method = "rmvpe"  # @param ["rmvpe", "crepe-full", "crepe-tiny", "fcpe", "harvest", "pyin"]
rvc_version = "v2"  # @param ["v1", "v2"]
cpu_cores = 2  # @param {type:"slider", min:1, max:8, step:1}
gpu_id = "0"  # @param {type:"string"}
embedder_model = "hubert_base"  # @param {type:"string"}
embedders_mode = "fairseq"  # @param ["fairseq", "transformers", "onnx", "whisper"]
f0_onnx = False  # @param {type:"boolean"}
predictor_onnx = False  # @param {type:"boolean"}
pitch_guidance = True  # @param {type:"boolean"}
hop_length = 128  # @param {type:"slider", min:32, max:512, step:32}
rms_extract = False  # @param {type:"boolean"}

import os
os.chdir(REPO_DIR)

cmd = f"python -m arvc.api.cli extract {model_name} --sample_rate {sample_rate}"
cmd += f" --f0_method {f0_method} --version {rvc_version}"
cmd += f" --cpu_cores {cpu_cores} --gpu {gpu_id}"
cmd += f" --embedder_model {embedder_model}"
cmd += f" --embedders_mode {embedders_mode}"
cmd += f" --hop_length {hop_length}"
if f0_onnx:
    cmd += " --f0_onnx"
if predictor_onnx:
    cmd += " --predictor_onnx"
if not pitch_guidance:
    cmd += " --no-pitch_guidance"
if rms_extract:
    cmd += " --rms_extract"

!{cmd}

In [ ]:
# @title Step 4: Create Index
# @markdown Create the .index file for voice retrieval.
model_name = ""  # @param {type:"string"}
rvc_version = "v2"  # @param ["v1", "v2"]
algorithm = "Auto"  # @param ["Auto", "Faiss", "KMeans"]

import os
os.chdir(REPO_DIR)

!python -m arvc.api.cli create-index {model_name} --version {rvc_version} --algorithm {algorithm}

In [ ]:
# @markdown ---
# @markdown ### 🏋️ Training Settings
# @title Step 5: Train Model
# @markdown Train the RVC voice model with full parameter control.
# @markdown **Important fixes applied**: gradient clipping enabled, LR warmup added, better default batch size.
# @markdown **Applio-parity accuracy** (commit f06a392+): per_preprocess=3.0 (~26% more chunks on small datasets),
# @markdown `--chunk_len`/`--overlap_len` now apply to Automatic cut mode too, model_info.json persists
# @markdown dataset_duration, and the saved .pth embeds `embedder_model` + `dataset_length` provenance fields.
# @markdown For best results on Colab T4: use batch_size=4, 200-300 epochs, rmvpe F0.
model_name = ""  # @param {type:"string"}
rvc_version = "v2"  # @param ["v1", "v2"]
epochs = 200  # @param {type:"slider", min:50, max:1000, step:50}
batch_size = 4  # @param ["2", "4", "8", "16"]
save_every = 25  # @param {type:"slider", min:5, max:100, step:5}
gpu_id = "0"  # @param {type:"string"}
author = ""  # @param {type:"string"}
optimizer = "AdamW"  # @param ["AdamW", "RAdam", "AnyPrecisionAdamW", "AdaBelief", "AdaBeliefV2"]
vocoder = "Default"  # @param ["Default", "MRF-HiFi-GAN", "RefineGAN", "BigVGAN"]
architecture = "RVC"  # @param ["RVC", "SVC"]
embedder_model = "hubert_base"  # @param {type:"string"}
embedders_mode = "fairseq"  # @param ["fairseq", "transformers", "onnx", "whisper"]
overtrain_detect = False  # @param {type:"boolean"}
overtrain_threshold = 50  # @param {type:"slider", min:10, max:100, step:5}
pitch_guidance = True  # @param {type:"boolean"}
cache_gpu = True  # @param {type:"boolean"}
checkpointing = False  # @param {type:"boolean"}
multiscale_loss = True  # @param {type:"boolean"}
cosine_lr = True  # @param {type:"boolean"}
energy_use = False  # @param {type:"boolean"}
deterministic = False  # @param {type:"boolean"}
benchmark = True  # @param {type:"boolean"}
compile_model = False  # @param {type:"boolean"}
fast_train = True  # @param {type:"boolean"}  # ~3x speedup (TF32+cuDNN+torch.compile+expandable_segments), vocal-quality-safe
bf16_adamw = False  # @param {type:"boolean"}  # Applio-parity shortcut: AnyPrecisionAdamW + bf16 autocast. Recommended on Ampere+ (A100/H100). Implies --optimizer=AnyPrecisionAdamW.
use_8bit_adam = False  # @param {type:"boolean"}
gradient_accumulation = 1  # @param {type:"slider", min:1, max:16, step:1}
use_reference = False  # @param {type:"boolean"}
reference_path = ""  # @param {type:"string"}
pretrained_g = ""  # @param {type:"string"}
pretrained_d = ""  # @param {type:"string"}

import os
os.chdir(REPO_DIR)

cmd = f"python -m arvc.api.cli train {model_name} --version {rvc_version}"
cmd += f" --epochs {epochs} --batch_size {batch_size}"
cmd += f" --save_every {save_every} --gpu {gpu_id}"
cmd += f" --optimizer {optimizer}"
cmd += f" --architecture {architecture}"
cmd += f" --embedder_model {embedder_model}"
cmd += f" --embedders_mode {embedders_mode}"
if vocoder != "Default":
    cmd += f" --vocoder {vocoder}"
if author:
    cmd += f" --author \"{author}\""
if overtrain_detect:
    cmd += f" --overtrain_detect --overtrain_threshold {overtrain_threshold}"
if not pitch_guidance:
    cmd += " --no-pitch_guidance"
if cache_gpu:
    cmd += " --cache_gpu"
if checkpointing:
    cmd += " --checkpointing"
if multiscale_loss:
    cmd += " --multiscale_loss"
if cosine_lr:
    cmd += " --cosine_lr"
if energy_use:
    cmd += " --energy"
if deterministic:
    cmd += " --deterministic"
if benchmark:
    cmd += " --benchmark"
if compile_model:
    cmd += " --compile_model"
if fast_train:
    cmd += " --fast_train"
if bf16_adamw:
    cmd += " --bf16_adamw"
if use_8bit_adam:
    cmd += " --use_8bit_adam"
if gradient_accumulation > 1:
    cmd += f" --gradient_accumulation {gradient_accumulation}"
if use_reference and reference_path:
    cmd += f" --use_reference --reference_path \"{reference_path}\""
if pretrained_g:
    cmd += f" --pretrained_g \"{pretrained_g}\""
if pretrained_d:
    cmd += f" --pretrained_d \"{pretrained_d}\""

!{cmd}

In [ ]:
# @markdown ---
# @markdown ### 🚀 One-Click Pipeline Settings
# @title One-Click Training (Full Pipeline)
# @markdown Run the entire training pipeline in one step: preprocess → extract → train → create index.
# @markdown **Uses optimized defaults** for Colab: batch_size=4, rmvpe F0, post-normalization, gradient clipping, LR warmup.
model_name = ""  # @param {type:"string"}
rvc_version = "v2"  # @param ["v1", "v2"]
sample_rate = "40k"  # @param ["32k", "40k", "48k"]
dataset_path = "./dataset"  # @param {type:"string"}
pitch_guidance = True  # @param {type:"boolean"}
f0_method = "rmvpe"  # @param ["rmvpe", "crepe-full", "crepe-tiny", "fcpe", "harvest", "pyin"]
total_epoch = 200  # @param {type:"slider", min:50, max:1000, step:50}
batch_size = 4  # @param ["2", "4", "8"]
save_every = 25  # @param {type:"slider", min:5, max:100, step:5}
gpu = "0"  # @param {type:"string"}
vocoder = "Default"  # @param ["Default", "MRF-HiFi-GAN", "RefineGAN", "BigVGAN"]
optimizer = "AdamW"  # @param ["AdamW", "RAdam", "AnyPrecisionAdamW", "AdaBelief", "AdaBeliefV2"]
embedder_model = "hubert_base"  # @param {type:"string"}
embedders_mode = "fairseq"  # @param ["fairseq", "transformers", "onnx", "whisper"]
model_author = ""  # @param {type:"string"}
architecture = "RVC"  # @param ["RVC", "SVC"]
cosine_lr = True  # @param {type:"boolean"}
multiscale_loss = True  # @param {type:"boolean"}
benchmark = True  # @param {type:"boolean"}
cache_gpu = True  # @param {type:"boolean"}
fast_train = True  # @param {type:"boolean"}  # ~3x speedup, vocal-quality-safe

import os, sys, subprocess
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Map "40k" → 40000 for CLI subprocess calls
sr_int = int(sample_rate.rstrip("k")) * 1000

# ── Step 1: Preprocess ──────────────────────────────────────────
print(f"▶ Step 1/4: Preprocessing '{model_name}'...")
preprocess_cmd = (
    f"python -m arvc.api.cli preprocess {model_name} --sample_rate {sr_int}"
    f" --dataset_path \"{dataset_path}\" --cpu_cores 2"
    f" --cut_method Automatic --process_effects"
    f" --chunk_len 3.0 --overlap_len 0.3 --normalization post"
)
ret = subprocess.run(preprocess_cmd, shell=True)
if ret.returncode != 0:
    print(f"❌ Preprocess failed (exit {ret.returncode}). Fix errors above before continuing.")
    raise SystemExit(ret.returncode)

# ── Step 2: Extract Features ────────────────────────────────────
print(f"\n▶ Step 2/4: Extracting features for '{model_name}'...")
extract_cmd = (
    f"python -m arvc.api.cli extract {model_name} --sample_rate {sr_int}"
    f" --version {rvc_version}"
    f" --f0_method {f0_method}"
    f" --cpu_cores 2 --gpu {gpu}"
    f" --embedder_model {embedder_model}"
    f" --embedders_mode {embedders_mode}"
    f" --hop_length 128"
)
if not pitch_guidance:
    extract_cmd += " --no-pitch_guidance"
ret = subprocess.run(extract_cmd, shell=True)
if ret.returncode != 0:
    print(f"❌ Extract failed (exit {ret.returncode}). Fix errors above before continuing.")
    raise SystemExit(ret.returncode)

# ── Step 3: Train ───────────────────────────────────────────────
print(f"\n▶ Step 3/4: Training '{model_name}' ({total_epoch} epochs)...")
train_cmd = f"python -m arvc.api.cli train {model_name} --version {rvc_version}"
train_cmd += f" --epochs {total_epoch} --batch_size {int(batch_size)}"
train_cmd += f" --save_every {save_every} --gpu {gpu}"
train_cmd += f" --optimizer {optimizer}"
train_cmd += f" --architecture {architecture}"
train_cmd += f" --embedder_model {embedder_model}"
train_cmd += f" --embedders_mode {embedders_mode}"
if vocoder != "Default":
    train_cmd += f" --vocoder {vocoder}"
if model_author:
    train_cmd += f" --author \"{model_author}\""
if not pitch_guidance:
    train_cmd += " --no-pitch_guidance"
if cache_gpu:
    train_cmd += " --cache_gpu"
if cosine_lr:
    train_cmd += " --cosine_lr"
if multiscale_loss:
    train_cmd += " --multiscale_loss"
if benchmark:
    train_cmd += " --benchmark"
if fast_train:
    train_cmd += " --fast_train"
ret = subprocess.run(train_cmd, shell=True)
if ret.returncode != 0:
    print(f"❌ Training failed (exit {ret.returncode}).")
    raise SystemExit(ret.returncode)

# ── Step 4: Create Index ────────────────────────────────────────
print(f"\n▶ Step 4/4: Creating index for '{model_name}'...")
index_cmd = f"python -m arvc.api.cli create-index {model_name} --version {rvc_version} --algorithm Auto"
ret = subprocess.run(index_cmd, shell=True)
if ret.returncode != 0:
    print(f"❌ Index creation failed (exit {ret.returncode}).")
    raise SystemExit(ret.returncode)

print(f"\n✅ One-click training pipeline complete for '{model_name}'!")
print(f"   Model weights: {REPO_DIR}/arvc/assets/logs/{model_name}/")

In [ ]:
# @title Create Reference Set
# @markdown Create a reference audio set for improved inference quality.
audio_file = ""  # @param {type:"string"}
ref_name = "reference"  # @param {type:"string"}
f0_method = "rmvpe"  # @param ["rmvpe", "crepe-full", "crepe-tiny", "fcpe", "harvest"]
pitch_shift = 0  # @param {type:"slider", min:-24, max:24, step:1}
rvc_version = "v2"  # @param ["v1", "v2"]
embedder_model = "hubert_base"  # @param {type:"string"}
embedders_mode = "fairseq"  # @param ["fairseq", "transformers", "onnx", "whisper"]
f0_autotune = False  # @param {type:"boolean"}
alpha = 0.5  # @param {type:"slider", min:0, max:1, step:0.01}

import os
os.chdir(REPO_DIR)

cmd = f"python -m arvc.api.cli create-ref \"{audio_file}\" -n {ref_name}"
cmd += f" --f0_method {f0_method} --version {rvc_version}"
cmd += f" --embedder_model {embedder_model}"
cmd += f" --embedders_mode {embedders_mode}"
if pitch_shift != 0:
    cmd += f" --pitch_shift {pitch_shift}"
if f0_autotune:
    cmd += " --f0_autotune"
if alpha != 0.5:
    cmd += f" --alpha {alpha}"

!{cmd}

<h2><font color="#3cba92">💾 Backup & Restore</font></h2>
<p>Backup models to Google Drive and restore them later.</p>


In [ ]:
# @title Backup Model to Drive
# @markdown Copy a trained model from Colab to your Google Drive for safekeeping.
model_name = "" # @param {type:"string"}

import os, shutil
from google.colab import drive
from google.colab._message import MessageError

try:
  drive.mount("/content/drive")
except MessageError:
  pass

src = f"{LOGS_PATH}/{model_name}"
dst = f"{BACKUPS_PATH}/{model_name}"

if not model_name:
    print("❌ Enter a model name.")
elif not os.path.exists(src):
    print(f"❌ Model '{model_name}' not found at {src}")
else:
    os.makedirs(BACKUPS_PATH, exist_ok=True)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"✅ Backed up '{model_name}' to Google Drive.")

In [ ]:
# @title List Backed Up Models

import os

if not os.path.exists(BACKUPS_PATH):
    print(f"❌ Backup directory not found: {BACKUPS_PATH}")
else:
    models = [d for d in os.listdir(BACKUPS_PATH) if os.path.isdir(os.path.join(BACKUPS_PATH, d))]
    if not models:
        print("No backed up models found.")
    else:
        for i, m in enumerate(models):
            size = sum(os.path.getsize(os.path.join(BACKUPS_PATH, m, f)) for f in os.listdir(os.path.join(BACKUPS_PATH, m)) if os.path.isfile(os.path.join(BACKUPS_PATH, m, f)))
            size_mb = size / (1024*1024)
            print(f"  {i+1}. {m} ({size_mb:.1f} MB)")

In [ ]:
# @title Restore Model from Drive
# @markdown Load a previously backed up model from Google Drive into Colab.
model_name = "" # @param {type:"string"}

import os, shutil

src = f"{BACKUPS_PATH}/{model_name}"
dst = f"{LOGS_PATH}/{model_name}"

if not model_name:
    print("❌ Enter a model name.")
elif not os.path.exists(src):
    print(f"❌ Backup '{model_name}' not found in Google Drive.")
else:
    os.makedirs(LOGS_PATH, exist_ok=True)
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f"✅ Restored '{model_name}' from Google Drive.")

<h2><font color="#ee0979">📤 Push to HuggingFace</font></h2>
<p>Upload your trained model to HuggingFace to share it with others.</p>


In [ ]:
# @title Push Model to HuggingFace
# @markdown Upload your trained model. Get a token from https://huggingface.co/settings/tokens
hf_token = "" #@param {type:"string"}
repo_id = "" #@param {type:"string"}
model_name = "" #@param {type:"string"}

import os, shutil
from huggingface_hub import HfApi, create_repo, login

login(hf_token)
api = HfApi()

try:
    create_repo(repo_id)
except Exception as e:
    print(f"Note: {e}. Proceeding...")

model_path = f"{LOGS_PATH}/{model_name}"
zip_path = f"{model_path}/{model_name}.zip"

if not os.path.exists(zip_path):
    print(f"Zipping {model_path}...")
    shutil.make_archive(f"{model_path}/{model_name}", 'zip', model_path)
    zip_path = f"{model_path}/{model_name}.zip"

api.upload_file(
    path_or_fileobj=zip_path,
    path_in_repo=f"{model_name}.zip",
    repo_id=repo_id,
    repo_type="model"
)

print(f"\n✅ Model uploaded to https://huggingface.co/{repo_id}")

<h2><font color="#9bc5c3">📖 CLI Quick Reference</font></h2>
